#### This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

train_dsdict = load_dataset('csv', data_files='train.csv')
train = train_dsdict['train']

#### **Multiple-Choice Data Formatting**
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

#### **Q1. Label Encoding**
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [ ]:
mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

def encode_labels(example):
    example['label'] = mapping[example['answer']]
    return example

# Map the labels across the dataset
train = train.map(encode_labels)

# Get the label for row index 150
answer_150 = train[150]['label']
print(f"Encoded numeric label for index 150: {answer_150}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Encoded numeric label for index 150: 2


#### **Q2. Prompt-Option Formatting**
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [ ]:
row_0 = train[0]
formatted_input = str(row_0['prompt']) + " [SEP] " + str(row_0['B'])
input_length = len(formatted_input)

print(f"Formatted string: {formatted_input}")
print(f"Character length: {input_length}")

Formatted string: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


#### **Tokenization for Multiple-Choice Models**
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences

#### **Q3. Single-Row MCQ Tokenization**
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:  
padding = "max_length"  
truncation = True  
max_length = 128  
return_tensors = "pt"  

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [ ]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Prepare the 5 inputs for row 0
options = ['A', 'B', 'C', 'D', 'E']
prompt = train[0]['prompt']
inputs = [str(prompt) + ' [SEP] ' + str(train[0][opt]) for opt in options]

# Tokenize
tokenized_inputs = tokenizer(
    inputs,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# Reshape to [batch_size, num_choices, seq_length] -> [1, 5, 128]
input_ids = tokenized_inputs['input_ids'].unsqueeze(0)

print(f"Input IDs shape: {input_ids.shape}")
print(f"Value of the second dimension: {input_ids.shape[1]}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input IDs shape: torch.Size([1, 5, 128])
Value of the second dimension: 5


#### **Q4. Batch MCQ Tokenization**
Tokenize the first 16 rows of train.csv as multiple-choice examples.  
Each row has 5 choices.  
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [ ]:
batch_rows = train.select(range(16))
options = ['A', 'B', 'C', 'D', 'E']

all_inputs = []
for row in batch_rows:
    for opt in options:
        all_inputs.append(str(row['prompt']) + ' [SEP] ' + str(row[opt]))

tokenized_batch = tokenizer(
    all_inputs,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# Reshape to [batch_size, num_choices, seq_length]
batch_input_ids = tokenized_batch['input_ids'].view(16, 5, 128)
total_positions = batch_input_ids.numel()

print(f"Tensor shape: {batch_input_ids.shape}")
print(f"Total token positions: {total_positions}")

Tensor shape: torch.Size([16, 5, 128])
Total token positions: 10240


#### **Multiple-Choice Model Outputs**
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

#### **Q5. Multiple-Choice Logits**
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [ ]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

# Prepare input for row 0 (already tokenized as 'input_ids' and 'tokenized_inputs' earlier)
with torch.no_grad():
    outputs = model(input_ids)

logits = outputs.logits
print(f"Logits shape: {logits.shape}")
print(f"Number of logits for one question: {logits.shape[1]}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Number of logits for one question: 5


#### **Q6. Supervised Loss Tensor**
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [ ]:
label = torch.tensor([train[0]['label']]) # The correct label for row 0

with torch.no_grad():
    outputs = model(input_ids, labels=label)

loss = outputs.loss
print(f"Loss value: {loss.item()}")
print(f"Number of dimensions in loss tensor: {loss.dim()}")

Loss value: 1.6171162128448486
Number of dimensions in loss tensor: 0


#### **LoRA for Efficient Fine-Tuning**
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

#### Q7. **LoRA Trainable Parameters**
Apply LoRA to the bert-base-uncased multiple-choice model using:  
r = 8    
lora_alpha = 16  
target_modules = ["query", "value"]  
lora_dropout = 0.1  
bias = "none"  
task_type = TaskType.SEQ_CLS  

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [ ]:
!pip uninstall -y torchao
!pip install -q peft
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForMultipleChoice
import torch

# Reloading model to ensure a clean state
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params}")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters: 295681


#### **Preparing Data for Hugging Face Trainer**
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

#### **Q8. Hugging Face Dataset Preparation**
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:  
input_ids with shape [5, 128]  
attention_mask with shape [5, 128]  
labels as the encoded answer label  

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [ ]:
from datasets import Dataset

# Take the first 100 rows
subset_train = train.select(range(100))

def preprocess_function(examples):
    first_sentences = [[prompt] * 5 for prompt in examples['prompt']]
    second_sentences = [[examples[opt][i] for opt in ['A', 'B', 'C', 'D', 'E']] for i in range(len(examples['prompt']))]

    # Flatten for tokenization
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=128,
        padding='max_length'
    )

    # Unflatten to [batch_size, 5, 128]
    return {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }

tokenized_dataset = subset_train.map(preprocess_function, batched=True)

# Check the shape of input_ids for the first item
first_input_ids = torch.tensor(tokenized_dataset[0]['input_ids'])
print(f"Shape of input_ids for first item: {first_input_ids.shape}")
print(f"Number of tokenized choices: {first_input_ids.shape[0]}")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Shape of input_ids for first item: torch.Size([5, 128])
Number of tokenized choices: 5


#### **Tiny Fine-Tuning and Inference**
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

#### **Q9. Tiny LoRA Fine-Tuning**
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:  
max_length = 64  
per_device_train_batch_size = 4  
gradient_accumulation_steps = 1  
max_steps = 4  

What is the final global_step reported by the Trainer?

In [ ]:
from transformers import TrainingArguments, Trainer

# Prepare subset for training (32 rows)
train_subset = tokenized_dataset.select(range(32))

# Define training arguments as per Q9
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to='none'
)

# Initialize Trainer
# Note: In newer versions, use processing_class instead of tokenizer if needed,
# but since the data is already tokenized, we can also pass it as processing_class.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    processing_class=tokenizer
)

# Run training
train_result = trainer.train()

print(f"Final global_step: {train_result.global_step}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.649344
2,1.665036
3,1.545236
4,1.631394


Final global_step: 4


In [ ]:
from transformers import TrainingArguments, Trainer

# Q9 spec requires max_length = 64 for fine-tuning.
# The Q8 dataset (tokenized_dataset) was built at max_length = 128 for Q8's own purposes,
# so we retokenize the first 32 rows here specifically at max_length = 64 for training.
train_raw_subset = train.select(range(32))

def preprocess_function_64(examples):
    first_sentences = [[p] * 5 for p in examples['prompt']]
    second_sentences = [[examples[opt][i] for opt in ['A', 'B', 'C', 'D', 'E']] for i in range(len(examples['prompt']))]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=64,
        padding='max_length'
    )

    return {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }

train_subset = train_raw_subset.map(preprocess_function_64, batched=True)

# Define training arguments as per Q9
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to='none'
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    processing_class=tokenizer
)

# Run training
train_result = trainer.train()

print(f"Final global_step: {train_result.global_step}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.764632
2,1.569078
3,1.628859
4,1.528752


Final global_step: 4


#### **Q10. Probability Assigned to Option E After Fine-Tuning**
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [ ]:
import torch.nn.functional as F

# Put model in evaluation mode
model.eval()

# Get the tokenized input for the first row (index 0)
# We'll use the input_ids we prepared earlier (shape [1, 5, 128])
# If it's on a specific device, we move it there, but here we are on CPU/Default
with torch.no_grad():
    outputs = model(input_ids)

# Apply softmax to the logits [1, 5]
probabilities = F.softmax(outputs.logits, dim=-1)

# Probability for Option E (index 4)
prob_E = probabilities[0, 4].item()

print(f"Probabilities for A-E: {probabilities[0].tolist()}")
print(f"Probability assigned to Option E: {prob_E:.4f}")

Probabilities for A-E: [0.19481047987937927, 0.20196546614170074, 0.20566008985042572, 0.19431902468204498, 0.2032449096441269]
Probability assigned to Option E: 0.2032


In [ ]:
import torch.nn.functional as F

# Ensure model is on the correct device and in eval mode
model.to(device)
model.eval()

# Retokenize row 0 at max_length = 64
options = ['A', 'B', 'C', 'D', 'E']
row0_inputs = [str(train[0]['prompt']) + ' [SEP] ' + str(train[0][opt]) for opt in options]

tokenized_row0 = tokenizer(
    row0_inputs,
    padding='max_length',
    truncation=True,
    max_length=64,
    return_tensors='pt'
)

# Move tensors to the same device as the model
input_ids_q10 = tokenized_row0['input_ids'].unsqueeze(0).to(device)  # [1, 5, 64]
attention_mask_q10 = tokenized_row0['attention_mask'].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(input_ids=input_ids_q10, attention_mask=attention_mask_q10)

# Apply softmax to the logits [1, 5]
probabilities = F.softmax(outputs.logits, dim=-1)

# Probability for Option E (index 4)
prob_E = probabilities[0, 4].item()

print(f"Probabilities for A-E: {probabilities[0].tolist()}")
print(f"Probability assigned to Option E: {prob_E:.4f}")

Probabilities for A-E: [0.19603630900382996, 0.20346270501613617, 0.20109352469444275, 0.20073065161705017, 0.1986767202615738]
Probability assigned to Option E: 0.1987
